# פרויקט 2: ניתוח ספקטרלי של הספק אלפא (Spectral Analysis of Alpha Power)

**קורס:** עיבוד אותות ביולוגיים  
**מגיש:** רועי כרמלי  

מחברת זו מציגה את הפתרון המלא לפרויקט 2 הכולל:
1. טעינה והצגה ויזואלית של אותות ה-EEG הגולמיים באלקטרודה POz.
2. סינון מעביר פסים (Band-Pass Filter) בטווח 2–40 הרץ.
3. אנליזה ספקטרלית והערכת תדר אלפא אינדיבידואלי (IAF) בשלושה אופנים (FFT, FFT מוחלק, שיטת Welch).
4. חישוב והצגת ספקטרוגרם (זמן-תדר).
5. שחזור אות אלפא מבודד באמצעות התמרת פורייה הפוכה (IFFT).

המחברת מותאמת להרצה מקומית וכן להרצה ישירה ב-Google Colab על ידי הורדה אוטומטית של קבצי הנתונים מה-GitHub במידת הצורך.

In [ ]:
# יבוא הספריות הנדרשות
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, welch, spectrogram

# הגדרת נתיב לשמירת גרפים במידה ומריצים מקומית
os.makedirs("graphs", exist_ok=True)
print("Libraries loaded successfully.")

In [ ]:
# פונקציה לטעינת הנתונים בצורה מותאמת ל-Google Colab
def load_subject_data(subject_id):
    filename = f"subject_{subject_id}.csv"
    if not os.path.exists(filename):
        # הורדה מה-Repository במידה וההרצה מתבצעת ב-Colab
        url = f"https://raw.githubusercontent.com/Royc4515/Project2_SignalProcessing/main/{filename}"
        print(f"Local file {filename} not found. Downloading from GitHub...")
        df = pd.read_csv(url)
    else:
        df = pd.read_csv(filename)
    return df

In [ ]:
# טעינת הנתונים עבור שלושת הנבדקים
fs = 1024  # תדר דגימה בהרץ
duration = 40  # אורך האות בשניות
t = np.linspace(0, duration, int(duration * fs), endpoint=False)

subjects = [504, 519, 524]
data = {}

for sub in subjects:
    df = load_subject_data(sub)
    data[subject_id := sub] = {
        'EC': df['EC'].values,
        'EO': df['EO'].values
    }
    print(f"Subject {sub} loaded: EC size = {len(df['EC'])}, EO size = {len(df['EO'])}")

## חלק 1: טעינה והצגה ויזואלית של אות ה-EEG

כאן נציג את האות הגולמי עבור תנאי עיניים פקוחות (EO) ועיניים עצומות (EC) במקביל, לאורך חלון זמן של 2 שניות (שניות 10 עד 12), על מנת להעריך ויזואלית את המחזוריות ואת נוכחותן של אוסצילציות אלפא.

In [ ]:
# הגדרת חלון זמן להצגה (2 שניות, למשל בין שנייה 10 לשנייה 12)
start_idx = int(10 * fs)
end_idx = int(12 * fs)

for sub in subjects:
    plt.figure(figsize=(12, 4.5))
    plt.plot(t[start_idx:end_idx], data[sub]['EO'][start_idx:end_idx], label='Eyes Open (EO)', color='#1f77b4', alpha=0.85)
    plt.plot(t[start_idx:end_idx], data[sub]['EC'][start_idx:end_idx], label='Eyes Closed (EC)', color='#ff7f0e', alpha=0.85)
    
    plt.title(f"Raw EEG Signal (POz Electrode) - Subject {sub}")
    plt.xlabel("Time (seconds)")
    plt.ylabel("Amplitude (μV)")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(f"graphs/subject_{sub}_raw.png", dpi=150)
    plt.show()

## חלק 2: סינון מעביר פסים (Band-Pass Filtering 2–40 Hz)

כדי להסיר רעשים בתדרים נמוכים מאוד (כגון סחיפה איטית של האלקטרודה וארטיפקטים של נשימה/תנועה) ורעשים בתדרים גבוהים (כמו רעשי שרירים ורעש רשת החשמל של 50 הרץ), נפעיל מסנן Butterworth מסדר 4 מעביר פסים בטווח 2–40 הרץ.

נשתמש ב-`scipy.signal.butter` לתכנון המסנן, וב-`scipy.signal.filtfilt` להפעלתו הדו-כיוונית. סינון דו-כיווני מונע עיוות פאזה (Zero-phase shift), מה ששומר על צורת האות המקורית ומאפשר השוואה אמינה בזמן.

In [ ]:
# פונקציה לסינון מעביר פסים
def bandpass_filter(data, fs, cutoff=[2, 40], order=4):
    nyquist = 0.5 * fs
    normal_cutoff = [c / nyquist for c in cutoff]
    b, a = butter(order, normal_cutoff, btype='bandpass', analog=False)
    y = filtfilt(b, a, data)
    return y

# סינון האותות לכל הנבדקים והתנאים
filtered_data = {}
for sub in subjects:
    filtered_data[sub] = {
        'EC': bandpass_filter(data[sub]['EC'], fs),
        'EO': bandpass_filter(data[sub]['EO'], fs)
    }

# הצגת קטע של 2 שניות (שניות 10 עד 12) המציג את האות הגולמי מול האות המסונן
for sub in subjects:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    
    # Eyes Open (EO)
    ax1.plot(t[start_idx:end_idx], data[sub]['EO'][start_idx:end_idx], label='Raw', color='gray', alpha=0.55)
    ax1.plot(t[start_idx:end_idx], filtered_data[sub]['EO'][start_idx:end_idx], label='Filtered (2-40 Hz)', color='#1f77b4', linewidth=1.5)
    ax1.set_title(f"Subject {sub} - Eyes Open (EO): Raw vs Filtered")
    ax1.set_ylabel("Amplitude (μV)")
    ax1.legend()
    ax1.grid(True, linestyle='--', alpha=0.5)
    
    # Eyes Closed (EC)
    ax2.plot(t[start_idx:end_idx], data[sub]['EC'][start_idx:end_idx], label='Raw', color='gray', alpha=0.55)
    ax2.plot(t[start_idx:end_idx], filtered_data[sub]['EC'][start_idx:end_idx], label='Filtered (2-40 Hz)', color='#ff7f0e', linewidth=1.5)
    ax2.set_title(f"Subject {sub} - Eyes Closed (EC): Raw vs Filtered")
    ax2.set_xlabel("Time (seconds)")
    ax2.set_ylabel("Amplitude (μV)")
    ax2.legend()
    ax2.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(f"graphs/subject_{sub}_filtered_comparison.png", dpi=150)
    plt.show()

## חלק 3: אנליזה ספקטרלית והערכת תדר אלפא אינדיבידואלי (IAF)

בסעיף זה נשווה בין שלושה אומדנים ספקטרליים שונים עבור האות המסונן:
1. **ספקטרום הספק מבוסס FFT:** חישוב ישיר באמצעות `np.fft.rfft` ונרמול לקבלת הספק (אמפליטודה בריבוע חלקי מספר הדגימות $N$). האומדן מניב רזולוציית תדר גבוהה מאוד (1/40 = 0.025 הרץ) אך סובל מווריאנס גבוה (רעש וקוצים רבים).
2. **ספקטרום FFT מוחלק (Smoothed FFT):** הפעלת מיצוע נע לאורך ציר התדר על ידי קונבולוציה של ספקטרום ה-FFT עם חלון משולש (Bartlett) בגודל 21 דגימות (השקול לרוחב פס של כ-0.5 הרץ). החלקה זו מפחיתה את הווריאנס של האומדן אך פוגעת ברזולוציית התדר.
3. **צפיפות הספק ספקטרלי בשיטת Welch:** חלוקת האות למקטעים עם חפיפה, הפעלת חלון Hamming על כל מקטע, חישוב ה-FFT וממוצע האומדנים. נבחר בחלון של 1024 דגימות (שנייה אחת) המניב רזולוציית תדר של 1 הרץ, עם חפיפה של 50% (512 דגימות). שיטה זו היא אומדן עקבי המפחית ווריאנס בצורה משמעותית (ע"י ממוצע).

**תדר אלפא אינדיבידואלי (IAF - Individual Alpha Frequency):**
נחשב את ה-IAF כנקודת התדר המקסימלית בטווח האלפא המוגדר (7.5–12.5 הרץ) לכל שיטה, נבדק ותנאי.

In [ ]:
# פונקציות עזר לחישובים ספקטרליים

def compute_fft_power(signal, fs):
    n = len(signal)
    freqs = np.fft.rfftfreq(n, d=1/fs)
    fft_vals = np.fft.rfft(signal)
    # ספקטרום הספק חד-צדדי (מנורמל באורך האות)
    power = (np.abs(fft_vals) ** 2) / n
    return freqs, power

def compute_smoothed_fft(power, window_size=21):
    # שימוש בחלון משולש Bartlett להחלקה לאורך ציר התדר
    window = np.bartlett(window_size)
    window = window / np.sum(window)  # נרמול שטח החלון ל-1
    return np.convolve(power, window, mode='same')

def compute_welch_psd(signal, fs, nperseg=1024, noverlap=512, window_type='hamming'):
    freqs, power = welch(signal, fs=fs, window=window_type, nperseg=nperseg, noverlap=noverlap)
    return freqs, power

def estimate_iaf(freqs, power):
    # חיתוך הטווח ל-7.5 עד 12.5 הרץ
    alpha_mask = (freqs >= 7.5) & (freqs <= 12.5)
    freqs_alpha = freqs[alpha_mask]
    power_alpha = power[alpha_mask]
    if len(power_alpha) == 0:
        return None
    max_idx = np.argmax(power_alpha)
    return freqs_alpha[max_idx]

In [ ]:
# הרצת האנליזה והדפסת תדרי ה-IAF
iaf_results = {}

for sub in subjects:
    iaf_results[sub] = {}
    for cond in ['EO', 'EC']:
        signal = filtered_data[sub][cond]
        
        # 1. FFT גולמי
        freqs_fft, power_fft = compute_fft_power(signal, fs)
        iaf_fft = estimate_iaf(freqs_fft, power_fft)
        
        # 2. FFT מוחלק
        power_smoothed = compute_smoothed_fft(power_fft, window_size=21)
        iaf_smoothed = estimate_iaf(freqs_fft, power_smoothed)
        
        # 3. שיטת Welch
        freqs_welch, power_welch = compute_welch_psd(signal, fs, nperseg=1024, noverlap=512, window_type='hamming')
        iaf_welch = estimate_iaf(freqs_welch, power_welch)
        
        iaf_results[sub][cond] = {
            'FFT': (freqs_fft, power_fft, iaf_fft),
            'Smoothed': (freqs_fft, power_smoothed, iaf_smoothed),
            'Welch': (freqs_welch, power_welch, iaf_welch)
        }
        
        print(f"Subject {sub} - {cond} | IAF (FFT): {iaf_fft:.3f} Hz | IAF (Smoothed): {iaf_smoothed:.3f} Hz | IAF (Welch): {iaf_welch:.3f} Hz")

In [ ]:
# יצירת גרף השוואתי בעל 3 תתי-איורים לכל נבדק
colors = {'EO': '#1f77b4', 'EC': '#ff7f0e'}
methods = ['FFT', 'Smoothed', 'Welch']
titles = [
    "Raw FFT Power Spectrum", 
    "Smoothed FFT Power Spectrum (Triangular Window, size=21)", 
    "Welch PSD (Hamming, nperseg=1024, noverlap=512)"
]

for sub in subjects:
    fig, axs = plt.subplots(3, 1, figsize=(11, 13), sharex=True)
    
    for i, method in enumerate(methods):
        ax = axs[i]
        for cond in ['EO', 'EC']:
            freqs, power, iaf = iaf_results[sub][cond][method]
            ax.plot(freqs, power, label=f"{cond} (IAF={iaf:.2f} Hz)", color=colors[cond], alpha=0.8)
            
            # סימון הפיק של ה-IAF
            if iaf is not None:
                closest_idx = np.argmin(np.abs(freqs - iaf))
                ax.scatter(iaf, power[closest_idx], color='red', s=50, zorder=5)
                ax.axvline(x=iaf, color=colors[cond], linestyle='--', alpha=0.45)
                
        ax.set_title(titles[i], weight='bold')
        ax.set_ylabel("Power (μV²/Hz)" if method == 'Welch' else "Power (μV²)")
        ax.legend(loc='upper right')
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.set_xlim(2, 30)  # הצגת טווח התדרים הרלוונטי בלבד
        
    axs[-1].set_xlabel("Frequency (Hz)")
    plt.suptitle(f"Spectral Estimations Comparison & IAF Detection - Subject {sub}", fontsize=14, weight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(f"graphs/subject_{sub}_spectral_estimates.png", dpi=150)
    plt.show()

## חלק 4: ספקטרוגרם (Spectrogram)

ספקטרוגרם הוא ייצוג זמן-תדר המציג כיצד ההספק של תדרים שונים משתנה לאורך זמן. הוא מבוסס על ביצוע STFT (Short-Time Fourier Transform) על מקטעי אות קצרים המשתנים בזמן.

נשתמש בפרמטרים הבאים:
- **אורך חלון (nperseg):** 512 דגימות (שווה ל-0.5 שניות). בחירה זו מאפשרת רזולוציית תדר טובה של 2 הרץ ורזולוציית זמן מצוינת לזיהוי שינויים מהירים.
- **סוג חלון:** חלון Hamming להפחתת זליגת אנרגיה ספקטרלית.
- **חפיפה (noverlap):** 256 דגימות (50% חפיפה) ליצירת רצף והחלקה לאורך הזמן.

ההספק יוצג בסקלה לוגריתמית (דציבלים - dB) כדי לאפשר הצגה ברורה של רכיבים חלשים וחזקים יחד.

In [ ]:
# חישוב והצגת ספקטרוגרמים לכל נבדק בתנאי EO ו-EC
for sub in subjects:
    plt.figure(figsize=(12, 8.5))
    
    for idx, cond in enumerate(['EO', 'EC']):
        plt.subplot(2, 1, idx+1)
        signal = filtered_data[sub][cond]
        
        # חישוב הספקטרוגרם
        f_spec, t_spec, Sxx = spectrogram(signal, fs=fs, window='hamming', nperseg=512, noverlap=256)
        
        # מסכת תדרים להצגת 2-30 הרץ בלבד
        freq_mask = (f_spec >= 2) & (f_spec <= 30)
        
        # המרה לדציבלים
        Sxx_db = 10 * np.log10(Sxx[freq_mask, :] + 1e-10)
        
        # ציור
        pcm = plt.pcolormesh(t_spec, f_spec[freq_mask], Sxx_db, shading='gouraud', cmap='jet')
        plt.title(f"Subject {sub} - {cond} Spectrogram (Hamming 0.5s window, 50% overlap)", weight='bold')
        plt.ylabel("Frequency (Hz)")
        plt.xlabel("Time (seconds)")
        plt.colorbar(pcm, label="Power (dB)")
        plt.ylim(2, 30)
        
    plt.tight_layout()
    plt.savefig(f"graphs/subject_{sub}_spectrogram.png", dpi=150)
    plt.show()

## חלק 5: שחזור אות אלפא באמצעות התמרת פורייה הפוכה (IFFT)

כדי לבודד את תנודת האלפא האינדיבידואלית של כל נבדק במישור הזמן, נבצע את השלבים הבאים:
1. נבחר את תדר ה-IAF המוערך באמצעות שיטת Welch (שהוכחה כיציבה והעקבית ביותר).
2. נגדיר את רצועת התדרים האינדיבידואלית (Individual Alpha Band) כ- **IAF ± 1 הרץ**.
3. נבצע FFT מלא דו-צדדי (`np.fft.fft`) של האות המסונן (מפרק 2).
4. נאפס לחלוטין את כל רכיבי התדר המורכבים (הן החיוביים והן השליליים כדי לשמור על הסימטריה ההרמיטית של האות במישור התדר) מחוץ לטווח המוגדר.
5. נשחזר את האות במישור הזמן באמצעות התמרת פורייה הפוכה (`np.fft.ifft`), וניקח את החלק הממשי (החלק המדומה שואף לאפס באופן נומרי).

In [ ]:
# ביצוע שחזור ה-IFFT
reconstructed_data = {}

for sub in subjects:
    reconstructed_data[sub] = {}
    for cond in ['EO', 'EC']:
        signal = filtered_data[sub][cond]
        
        # שליפת תדר ה-Welch IAF
        iaf = iaf_results[sub][cond]['Welch'][2]
        
        # FFT דו-צדדי
        N_samples = len(signal)
        Y = np.fft.fft(signal)
        freqs = np.fft.fftfreq(N_samples, d=1/fs)
        
        # בניית מסנן במישור התדר (פס מעבר סביב ה-IAF)
        alpha_mask = (np.abs(freqs) >= (iaf - 1)) & (np.abs(freqs) <= (iaf + 1))
        
        # איפוס תדרים מחוץ לטווח
        Y_alpha = Y.copy()
        Y_alpha[~alpha_mask] = 0.0
        
        # שחזור על ידי IFFT
        reconstructed = np.fft.ifft(Y_alpha).real
        reconstructed_data[sub][cond] = reconstructed

In [ ]:
# הצגת האות המשוחזר מול האות המסונן המקורי לאורך 2 שניות (שניות 10 עד 12)
for sub in subjects:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7.5), sharex=True)
    
    # Eyes Open (EO)
    ax1.plot(t[start_idx:end_idx], filtered_data[sub]['EO'][start_idx:end_idx], label='Original Filtered (2-40 Hz)', color='gray', alpha=0.55)
    ax1.plot(t[start_idx:end_idx], reconstructed_data[sub]['EO'][start_idx:end_idx], label='Reconstructed Alpha (IAF ± 1 Hz)', color='#1f77b4', linewidth=1.8)
    ax1.set_title(f"Subject {sub} - Eyes Open (EO): Original vs Reconstructed Alpha")
    ax1.set_ylabel("Amplitude (μV)")
    ax1.legend(loc='upper right')
    ax1.grid(True, linestyle='--', alpha=0.5)
    
    # Eyes Closed (EC)
    ax2.plot(t[start_idx:end_idx], filtered_data[sub]['EC'][start_idx:end_idx], label='Original Filtered (2-40 Hz)', color='gray', alpha=0.55)
    ax2.plot(t[start_idx:end_idx], reconstructed_data[sub]['EC'][start_idx:end_idx], label='Reconstructed Alpha (IAF ± 1 Hz)', color='#ff7f0e', linewidth=1.8)
    ax2.set_title(f"Subject {sub} - Eyes Closed (EC): Original vs Reconstructed Alpha")
    ax2.set_xlabel("Time (seconds)")
    ax2.set_ylabel("Amplitude (μV)")
    ax2.legend(loc='upper right')
    ax2.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(f"graphs/subject_{sub}_reconstructed_alpha.png", dpi=150)
    plt.show()